In [ ]:
import os
import pathlib

import geomulticorr as gmc
import geoutils as gu

import matplotlib.pyplot as plt

In [ ]:
session = gmc.open_gmc_session(target_directory_path=pathlib.Path('/home/cusicand/03_Data/GMC_projects/MultiSensor'), epsg_code=2154, empty_geodatabase=True)

In [ ]:
raster_bank = session.map_georasters_bank(f"/home/cusicand/05_Devs/GeoMultiCorr/geomulticorr/data/french_alps/",
                                          epsg_code=2154)

In [ ]:
raster_bank = session.get_georasters_map()

In [ ]:
raster_bank

In [ ]:
raster_bank.plot(column="sensor", legend=True, facecolor="none")

## Drawn PZONE in QGIS or in JupyterNotebook

In [ ]:
# ! To be tested with a polygon drawn on the map:
get_gdf = session.draw_polygon_manually()

In [ ]:
# draw on the map, then:
aoi = get_gdf()

In [ ]:
session.insert_pzone(aoi.geometry.iloc[0], pz_name="MontVallon", pz_shortname="MV")

In [ ]:
session.update_vector_data_session()

In [ ]:
session._pzones

In [ ]:
selected_rasters = raster_bank[raster_bank["sensor_family"].isin(["planetscope"])]

In [ ]:
selected_rasters

In [ ]:
session.sieve_bulk(selected_rasters,
                   target_resolution=3)                #    canonical_bounds=canon_bounds,
                #    canonical_grid_size=canon_size)

In [ ]:
session.update_thumbs()

In [ ]:
session.validate_all_thumbs()

In [ ]:
session.update_pairs_with_strategy(strategy="step", max_step=1)

In [ ]:
from geomulticorr.correlation import ASP

asp = ASP()


In [ ]:
scripts = session.prepare_pairs_correlation(
    cluster="local",
    cores=16,
    corr_algorithm="asp_bm",
    corr_kernel=(21, 21),
)

In [ ]:
scripts

In [ ]:
print(scripts[0].read_text())   # see the generated OAR bash script

In [ ]:
# Dry-run to preview launch commands
session.launch_pairs_correlation(cluster="local", dry_run=True)

In [ ]:
ret = session.launch_pairs_correlation(
    criterias="MontVallon_2017-08-01-planetscope_2018-08-26-planetscope",
    corr_algorithm="asp_bm",
    processes=4,
)

In [ ]:
# Local execution
session.launch_pairs_correlation()

# Debug manually

In [ ]:
import pathlib
import geoutils as gu
import re
import datetime
from geomulticorr import extract_acquisition_date
import pandas as pd
import geopandas as gpd

In [ ]:
workdir = pathlib.Path('/home/cusicand/05_Devs/GeoMultiCorr/geomulticorr/data/french_alps/')
print(workdir)

In [ ]:
list_rasters = list(workdir.glob("**/*.tif"))
print(f"Found {len(list_rasters)} rasters.")

In [ ]:
list_rasters

In [ ]:
single_raster = list_rasters[0]
print(f"Testing raster: {single_raster}")

In [ ]:
r = gu.Raster(single_raster, load_data=False)

In [ ]:
supported_sensors = [
        # Landsat
        "landsat4", "landsat5", "landsat7", "landsat8", "landsat9",
        "l4", "l5", "l7", "l8", "l9", "lt04", "lt05", "lc08", "lc09", "le07",
        # Sentinel-2 (MSI) # May be some problems with "T32TLR", "T31TGL", targets.
        "sentinel2", "sentinel-2", "s2", "s2a", "s2b", "msi", "T32TLR", "T31TGL",
        # SPOT / Airbus VHR
        "spot4", "spot5", "spot6", "spot7", "sp4", "sp5", "sp6", "sp7", "s4p", "s5p", "s6p", "s7p",
        "pleiades", "pléiades", "pleiades-neo", "pneo", "ple",
        # Planet
        "planetscope", "planet", "psscene", "superdove", "ps2", "ps2sd", "pscope", "planet"
        # Maxar family
        "worldview", "wv1", "wv2", "wv3", "wv4", "geoeye1", "ge1", "quickbird", "ikonos",
        # RapidEye
        "rapideye", "re1", "re2", "re3", "re4", "re5",
        # Aerial / DEM
        "aerial", "uav", "drone", "swissimage", "dem", "hillshade",
    ]

In [ ]:
def re_searcher(string: str, pattern_for_search: str) -> str:
    """Search for a pattern in a string and return the first match.

    Args:
        string (str): The string to search in.
        pattern (str): The pattern to search for.

    Returns:
        str: The first match within the string.
    """
    try:
        return re.search(re.compile(pattern_for_search), string).group()
    except AttributeError:
        return "unknown"
    #END try
#END def

In [ ]:
def sensors(sensors_names: list = None) -> str:
    s = ""
    for string in sensors_names:
        s += string.lower() + "|"
        s += string.upper() + "|"
    s = s[:-1]
    return s

In [ ]:
def sensor_normalize(text) -> dict:
    """Normalize sensor names to a canonical form.

    Args:
        text (Optional[str]): The input sensor name.

    Returns:
        dict: A dictionary with normalized sensor information.
    # * Works properly
    """
    if not text:
        return {"sensor": "unknown", "platform": None}

    t = text.lower()

    def out(sensor, *, platform=None, family=None, conf=0.95, matched=None):
        return {"sensor": sensor, "platform": platform, "family": family}
    
    # --- Sentinel-2 (S2A/S2B or generic)
    m = re.search(r"\b(sentinel[-\s]?2|s2)\s*([ab])?\b|\bsen2*([ab])?\b|\bsent2*([ab])?\b|\b[A-Za-z]{1}\d{2}[A-Za-z]{3}\b|\b[A-Za-z]{1}\d{2}[A-Za-z]{3}\b", t)
    if m:
        var = m.group(2).upper() if m.group(2) else None
        plat = f"sentinel-{var}" if var else None
        return out("sentinel-2", platform=plat, family="sentinel")

    # --- Landsat (L5/L7/L8/L9 codes; classic names)
    # m = re.search(r"\b(?:landsat\s*(5|7|8|9)|l(5|7|8|9))\b", t)
    m = re.search(r"\b(?:landsat\s*(4|5|7|8|9)|l(4|5|7|8|9)|lc0?(4|5|7|8|9)|le0?(4|5|7|8|9)|lt0?(4|5|7|8|9))\b", t)
    if m:
        num = next(g for g in m.groups() if g)
        return out(f"landsat-{num}", platform=f"landsat-{num}", family="landsat")

    # --- SPOT (4/5/6/7)
    # m = re.search(r"\bspot\s*(4|5|6|7)\b", t)
    m = re.search(r"\bspot\s*(4|5|6|7)\b|\bsp\s*(4|5|6|7)\b|\bs\s*(4|5|6|7)p\b", t)
    if m:
        # num = m.group(1)
        num = next(g for g in m.groups() if g)
        return out(f"spot-{num}", platform=f"spot-{num}", family="spot")

    # --- Pléiades 1A/1B
    m = re.search(r"\b(pl[eé]iades)\s*(1a|1b)\b|\bple\b|\bpl1[ab]\b|\bphr[1-2]a\b|\bphr[1-2]b\b", t)
    if m:
        var = m.group(2).lower()
        return out("pleiades", platform=f"pleiades-{var}", family="pleiades")

    # --- Pléiades Neo (Neo1/Neo2)
    m = re.search(r"\b(pl[eé]iades[-\s]?neo|pneo)\s*(\d+)?\b", t)
    if m:
        var = m.group(2)
        plat = f"pleiades-neo-{var}" if var else None
        return out("pleiades-neo", platform=plat, family="pleiades")

    # --- PlanetScope (PS2, PS2.SD / SuperDove, PSScene)
    m = re.search(r"\b(superdove|ps2\.?sd|psscene|planetscope|ps2|psb|planet)\b", t)
    if m:
        token = m.group(1)
        var = "SuperDove" if token in {"superdove", "ps2.sd", "ps2sd"} else None
        return out("planetscope", platform="planetscope", family="planetscope")

    # --- RapidEye (RE1..RE5)
    m = re.search(r"\brapideye\b|\bre([1-5])\b", t)
    if m:
        var = m.group(1).lower() if m.group(1) else None
        plat = f"rapideye-{var}" if var else None
        return out("rapideye", platform=plat, family="rapideye")

    # --- WorldView (WV1..WV4)
    m = re.search(r"\b(?:worldview[-\s]?(1|2|3|4)|wv(1|2|3|4))\b", t)
    if m:
        num = next(g for g in m.groups() if g)
        return out(f"worldview-{num}", platform=f"worldview-{num}", family="worldview")

    # --- GeoEye-1
    m = re.search(r"\bgeoeye[-\s]?1\b|\bge1\b", t)
    if m:
        return out("geoeye-1", platform="geoeye-1", family="geoeye")

    # --- QuickBird
    m = re.search(r"\bquickbird\b|\bqb[12]?\b", t)
    if m:
        return out("quickbird", platform="quickbird-2", family="quickbird")

    # --- IKONOS
    m = re.search(r"\bikonos\b", t)
    if m:
        return out("ikonos", platform="ikonos", family="ikonos")

    # --- Aerial / UAV / SwissImage
    if re.search(r"\bswissimage\b", t):
        return out("aerial", platform="swissimage", family="aerial")#, matched="swissimage")
    if re.search(r"\b(aerial|orthophoto|uav|drone)\b", t):
        return out("aerial", platform="aerial", family="aerial")#, matched="aerial/uav/drone")

    # --- DEM / hillshade
    if re.search(r"\bdem\b|\bhillshade\b", t):
        return out("dem", platform=None, family="dem")#, matched="dem/hillshade")

    # Unknown
    return out("unknown", conf=0.0, matched=None)

In [ ]:
georaster_metadata = {
            "filename": None,
            "sensor": None,
            "xRes": None,
            "yRes": None,
            "bands": None,
            "rows": None,
            "cols": None,
            "filepath": None,
            "acq_date": None,
            "file_path": None,
            "sensor_family": None,
            "sensor_platform": None,
            "acq_datetime": None,
            "src_crs": None,
            "geometry": None,
}
georaster_metadata

In [ ]:
def search_date_in_filename(filename: str | pathlib.Path) -> str | None:
    """Find the first date in a filename and normalize to YYYY-MM-dd.

    Supports the following inputs:
    - YYYYmmdd
    - YYYY-mm-dd
    - YYYY/mm/dd
    - DD-MM-YYYY
    - DD/MM/YYYY
    Returns None if no date is found.
    """
    # Ordered list of (regex, strptime_format) pairs
    patterns = [
        # YYYYmmdd
        (r"(19\d{2}|20\d{2}|21\d{2})(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])", "%Y%m%d"),
        # YYYY-mm-dd
        (r"(19\d{2}|20\d{2}|21\d{2})-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])", "%Y-%m-%d"),
        # YYYY/mm/dd
        (r"(19\d{2}|20\d{2}|21\d{2})/(0[1-9]|1[0-2])/(0[1-9]|[12]\d|3[01])", "%Y/%m/%d"),
        # DD-MM-YYYY
        (r"(0[1-9]|[12]\d|3[01])-(0[1-9]|1[0-2])-(19\d{2}|20\d{2}|21\d{2})", "%d-%m-%Y"),
        # DD/MM/YYYY
        (r"(0[1-9]|[12]\d|3[01])/(0[1-9]|1[0-2])/(19\d{2}|20\d{2}|21\d{2})", "%d/%m/%Y"),
        # YYYYmmddhhmmss (14 digits)
        (r"(19\d{2}|20\d{2}|21\d{2})(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])([01]\d|2[0-3])([0-5]\d){2}", "%Y%m%d%H%M%S"),
        # YYYYmmddThhmmss (with 'T' separator)
        (r"(19\d{2}|20\d{2}|21\d{2})(0[1-9]|1[0-2])([0-2]\d|3[01])T([01]\d|2[0-3])([0-5]\d){2}", "%Y%m%dT%H%M%S"),
    ]

    s = str(filename)
    for pat, fmt in patterns:
            m = re.search(pat, s)
            if not m:
                continue
            raw = m.group(0)
            try:
                dt = datetime.strptime(raw, fmt)
                return dt.strftime("%Y-%m-%d")
            except ValueError:
                # If 14-digit or T pattern fails, try first 8 digits as date
                if fmt in ("%Y%m%d%H%M%S", "%Y%m%dT%H%M%S") and len(raw) >= 8:
                    try:
                        dt = datetime.strptime(raw[:8], "%Y%m%d")
                        return dt.strftime("%Y-%m-%d")
                    except ValueError:
                        continue
                continue
    return None

In [ ]:
patterns = [
    # YYYYmmdd
    (r"(19\d{2}|20\d{2}|21\d{2})(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])", "%Y%m%d"),
    # YYYY-mm-dd
    (r"(19\d{2}|20\d{2}|21\d{2})-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])", "%Y-%m-%d"),
    # YYYY/mm/dd
    (r"(19\d{2}|20\d{2}|21\d{2})/(0[1-9]|1[0-2])/(0[1-9]|[12]\d|3[01])", "%Y/%m/%d"),
    # DD-MM-YYYY
    (r"(0[1-9]|[12]\d|3[01])-(0[1-9]|1[0-2])-(19\d{2}|20\d{2}|21\d{2})", "%d-%m-%Y"),
    # DD/MM/YYYY
    (r"(0[1-9]|[12]\d|3[01])/(0[1-9]|1[0-2])/(19\d{2}|20\d{2}|21\d{2})", "%d/%m/%Y"),
    # YYYYmmddhhmmss (14 digits)
    (r"(19\d{2}|20\d{2}|21\d{2})(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])([01]\d|2[0-3])([0-5]\d){2}", "%Y%m%d%H%M%S"),
    # YYYYmmddThhmmss (with 'T' separator)
    (r"(19\d{2}|20\d{2}|21\d{2})(0[1-9]|1[0-2])([0-2]\d|3[01])T([01]\d|2[0-3])([0-5]\d){2}", "%Y%m%dT%H%M%S"),
]

In [ ]:
list_rasters[0]

In [ ]:
s = str(list_rasters[0])
for pat, fmt in patterns:
        m = re.search(pat, s)
        print(m)

In [ ]:
target_crs = "2154"
for ta in list_rasters:
    # Openning raster with geoutils to access metadata without loading full data into memory
    r = gu.Raster(ta, load_data=False)
    georaster_metadata["filename"] = ta.name
    georaster_metadata["file_path"] = str(ta)
    georaster_metadata["xRes"] = r.res[0]
    georaster_metadata["yRes"] = r.res[1]
    georaster_metadata["bands"] = r.count
    georaster_metadata["rows"] = r.shape[0]
    georaster_metadata["cols"] = r.shape[1]
    sensor = re_searcher(ta.name.lower(), sensors(supported_sensors))
    print(f"Raster: {ta.name}")
    print(f"Sensor extracted: {sensor}")
    sensor_normalized = sensor_normalize(sensor)
    georaster_metadata["sensor_platform"] = sensor_normalized["platform"]
    georaster_metadata["sensor_family"] = sensor_normalized["family"]
    # Struggling to extract acquisition date, may be due to sensor-specific formats or missing metadata
    georaster_metadata["acq_datetime"] = extract_acquisition_date(ta, sensor=sensor)
    georaster_metadata["acq_date"] = georaster_metadata["acq_datetime"].date()

    date_dt = extract_acquisition_date(ta)
    print(f"Extracted datetime: {date_dt}")

    fp = r.footprint
    if hasattr(fp, "ds"):
        gdf_fp = fp.ds
        src_crs = gdf_fp.crs
        shp = gdf_fp.geometry.union_all()
    else:
        shp = fp
        src_crs = getattr(r, "crs", None)

    georaster_metadata["src_crs"] = (src_crs.to_string() if hasattr(src_crs, "to_string") else str(src_crs)) if src_crs else None

    geom_proj = shp
    try:
        if target_crs and src_crs and str(src_crs) != target_crs:
            geom_proj = gpd.GeoSeries([shp], crs=src_crs).to_crs(target_crs).iloc[0]
    except Exception:
        pass

    georaster_metadata["geometry"] = geom_proj  # serialize for cross-process return

In [ ]:
georaster_metadata

In [ ]:
r.footprint